## E3: Can PerturbOT predict the expression of out-of-sample cell types or subtypes excluded during the Gromov step?

**Started:** October 23, 2025 

**Last updated:** October 23, 2025

**Research question:** Can PerturbOT predict the expression of out-of-sample cell types or subtypes excluded during Gromov mapping?

**Hypothesis:** PerturbOT will be able to predict the expression of cells close to other cell types (e.g., CD4 T cells when other T cell are included in the Gromov step), but will not be able to predict the expression of cell types not well represented in the training set.

**Conclusion:** This approach cannot predict OOS for hematopoeitic stem cells, regulatory T cells, or mast cells. I did not find any cell types this works for.

**Potential Next Steps:** Try with low-entryopy (1e-8) GW Perturb-OT.

In [1]:
import numpy as np
from sklearn.decomposition import PCA
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy as sp
import umap
import matplotlib.pylab as pl
import matplotlib.pyplot as plt
import seaborn as sns
import random
from speciesot_helpers import label_uniform_cell_type_row, \
                              top_n_organisms_from_species, \
                              cell_types_with_n_per_organism, \
                              sample_equal_cell_types, \
                              ot_between_organisms, \
                              calc_random_mean_std, \
                              test_train_split_adata, \
                              get_cell_type_from_index, \
                              create_label_to_sample_dicts, \
                              concat_index, \
                              show_cell_type_results, \
                              cell_type_knn_acc, \
                              plot_predictions
from pytorch_helpers import fit_mlp
import perturbot
from perturbot.match import get_coupling_egw_labels_ott
from perturbot.predict import train_mlp

/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/lightning/fabric/__init__.py:40: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/anndata/utils.p

In [2]:
human_dir = 'data/tabula_sapiens/'
mouse_dir = 'data/tabula_muris/'

In [3]:
human_adatas = top_n_organisms_from_species(human_dir, 8, 'human')

In [4]:
mouse_adatas = top_n_organisms_from_species(mouse_dir, 8, 'mouse')

In [5]:
all_adatas = mouse_adatas + human_adatas
pairs = [(i,i+8) for i in range(8)]

In [28]:
oos_cell_type = 'NK cell'

In [45]:
adata[adata.obs['cell_type']=='NK cell']

NameError: name 'adata' is not defined

In [67]:
# which adatas have best representation of the desired cell type?
best_human_adata, best_mouse_adata = human_adatas[0], mouse_adatas[0]

for adata in human_adatas[1:]:
    num_desired_cell_type = len(adata[adata.obs['shared_cell_type']=='NK cell'])
    if num_desired_cell_type > len(best_human_adata[best_human_adata.obs['shared_cell_type']=='NK cell']):
        best_human_adata = adata
        
for adata in mouse_adatas[1:]:
    num_desired_cell_type = len(adata[adata.obs['shared_cell_type']=='NK cell'])
    if num_desired_cell_type > len(best_mouse_adata[best_mouse_adata.obs['shared_cell_type']=='NK cell']):
        best_mouse_adata = adata

13
49
112
66
26
40
16


In [29]:
# Gather data
n_samples = 2000
a1_full, a2_full = sample_equal_cell_types(all_adatas[pairs[0][0]], all_adatas[pairs[0][1]], n_samples, exact_cell_type_match=oos_cell_type)

In [69]:
a1_oos_all = best_human_adata[best_human_adata.obs['shared_cell_type']==oos_cell_type]
a2_oos_all = best_mouse_adata[best_mouse_adata.obs['shared_cell_type']==oos_cell_type]

In [84]:
num_oos = np.min([len(a1_oos_all), len(a2_oos_all)])

In [90]:
a1_oos_all_equal = a1_oos_all[pd.Series(a1_oos_all.obs_names).sample(frac=1)[:num_oos].tolist()]
a2_oos_all_equal = a2_oos_all[pd.Series(a2_oos_all.obs_names).sample(frac=1)[:num_oos].tolist()]

In [30]:
a1_train, a1_test, a2_train, a2_test = test_train_split_adata(a1_full, a2_full, train_ratio=0.8)

In [31]:
a1_train = a1_full[a1_full.obs['cell_type']!=oos_cell_type]
a1_test = a1_oos_all_equal
a2_train = a2_full[a2_full.obs['cell_type']!=oos_cell_type]
a2_test = a2_full[a2_full.obs['cell_type']==oos_cell_type]

In [ ]:
a1_test = 

In [44]:
a1_train

View of AnnData object with n_obs × n_vars = 2000 × 18024
    obs: 'batch', 'tissue_FACS_droplet', 'free_annotation', 'n_counts', 'n_genes', 'louvain', 'leiden', 'age', 'method', 'donor_id', 'subtissue', 'tissue_free_annotation', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'suspension_type', 'FACS.selection', 'tissue_original', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'shared_cell_type'
    var: 'n_cells-0', 'n_cells-1', 'means', 'dispersions', 'dispersions_norm', 'highly_variable', 'gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'age_colors', 'citation', 'leiden', 'louvain', 'louvain_colors', 'method_colors', 'neighb

In [32]:
X_dict, Y_dict, X_full_dict, Y_full_dict, X_index_dict, Y_index_dict = create_label_to_sample_dicts(a1_train, a2_train)

In [33]:
# Learn matching in the latent space
T_dict, log = get_coupling_egw_labels_ott((X_dict, Y_dict), 1e-8) # Other get_coupling_X methods be used

running EGWL with ott
GW called
lse step
updating linearization
Label considered for Sinkhorn run
lse step
updating linearization
Label considered for Sinkhorn run
5 outer iterations were needed.
The last Sinkhorn iteration has converged: False
The outer loop of Gromov Wasserstein has converged: False
The final regularized GW cost is: nan
Done running LEGWOT with ott


In [38]:
Y_paired_dict = {}
for i in range(len(T_dict)):
    pairings_i = np.argmax(T_dict[i], axis=1)
    Y_paired_dict[i] = np.zeros(Y_dict[i].shape)
    for j in range(len(pairings_i)):
        Y_paired_dict[i][j] = Y_dict[i][pairings_i[j]]

X_paired_stacked = np.vstack([X_dict[k] for k in X_dict])  
Y_paired_stacked = np.vstack([Y_paired_dict[k] for k in Y_paired_dict])

result = fit_mlp(X_paired_stacked.astype(np.float32), 
                 Y_paired_stacked.astype(np.float32), 
                 epochs=500, 
                 batch_size=128, 
                 lr=1e-3)
model_paired = result['model']

Epoch 001 | loss 15.476594
Epoch 100 | loss 0.816383
Epoch 200 | loss 0.494689
Epoch 300 | loss 0.446594
Epoch 400 | loss 0.391797
Epoch 500 | loss 0.355080


In [42]:
X_test_dict

{}

In [39]:
X_test_dict, Y_test_dict, X_test_full_dict, Y_test_full_dict, X_test_index_dict, Y_test_index_dict = create_label_to_sample_dicts(a1_test, a2_test)
sorted_test_index = concat_index([Y_test_index_dict[k] for k in Y_test_dict])
X_test_stacked = np.vstack([X_test_dict[k] for k in X_test_dict])  
Y_test_stacked = np.vstack([Y_test_dict[k] for k in Y_test_dict])  

ValueError: need at least one array to concatenate

In [40]:
X_paired_stacked_all = np.vstack([X_dict[k] for k in X_dict] + [X_test_dict[k] for k in X_test_dict])  
Y_paired_stacked_all = np.vstack([Y_paired_dict[k] for k in Y_paired_dict] + [Y_test_dict[k] for k in Y_test_dict])

In [41]:
sorted_index_all = concat_index([Y_index_dict[k] for k in Y_full_dict] + [Y_test_index_dict[k] for k in Y_test_dict])

In [ ]:
cell_type_knn_acc(model_paired, X_test_stacked, Y_paired_stacked_all, all_adatas[pairs[0][1]], sorted_index_all, sorted_test_index)

In [ ]:
model = model_paired
X_eval = X_test_stacked
Y_reference = Y_paired_stacked_all
target_adata = all_adatas[pairs[0][1]]
sorted_reference_index =  sorted_index_all
sorted_test_index = sorted_test_index
n_neighbors=1

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy as sp
import matplotlib.pylab as pl
import matplotlib.pyplot as plt
import ot
import random
import torch
from sklearn.neighbors import NearestNeighbors

In [ ]:
model.eval()
device = next(model.parameters()).device
with torch.inference_mode():
    preds = model(torch.from_numpy(X_eval).to(device).float()).cpu().numpy()

nbrs = NearestNeighbors(n_neighbors=n_neighbors, algorithm='ball_tree').fit(Y_reference)

reference_distances, reference_indices = nbrs.kneighbors(preds)
cell_type_results_from_reference = show_cell_type_results(reference_indices, sorted_reference_index, target_adata)

cell_type_result_list = []
for i in range(len(cell_type_results_from_reference)):
    consensus_ct = max(set(cell_type_results_from_reference[i]), key=cell_type_results_from_reference[i].count)
    cell_type_result_list.append(consensus_ct)

pred_ct_series = pd.Series(cell_type_result_list, index=sorted_test_index,)
true_ct_series = target_adata[sorted_test_index,].obs['shared_cell_type'] 

baseline_acc = 100 * sum(true_ct_series == pred_ct_series) / len(pred_ct_series)

In [ ]:
true_ct_series

In [ ]:
pred_ct_series

### Plot true values and predictions on PCA and UMAP

In [ ]:
plot_predictions(a2_train, a2_test, preds, oos_cell_type)